# Topic D — L4: Implicit & Crank-Nicolson Finite Differences

*FINM 32000 Numerical Methods — Final-Exam Study Notes*

---

**Prerequisites:** [Topic C — Explicit FD](Topic-C_L3_ExplicitFiniteDifferences.ipynb) (grid, finite differences, stability of explicit scheme).

**What this notebook covers**
1. Why implicit methods exist (fix explicit stability limit)
2. Fully implicit FD scheme and tridiagonal systems
3. **Crank-Nicolson** — the scheme used in class for production pricing
4. Solving tridiagonal systems (Thomas algorithm / `numpy.linalg.solve`)
5. Greeks and **charm** from the grid (Practice-Midterm P4c)
6. Americans on FD grids; Breeden–Litzenberger via strike derivatives

**Exam map**

| Source | What it tests |
|---|---|
| L4.4–L4.7 | Implicit scheme; unconditional stability |
| L4.12–L4.15 | Crank-Nicolson formula; $O(\Delta t^2)$ accuracy |
| Practice-Midterm P4(b) | Reconstruct hidden grid value from CN equation |
| Practice-Midterm P4(c) | Charm $\partial^2 C/(\partial t\,\partial S)$ from grid |
| HW4 P2(b) | Crank-Nicolson American put (CEV) |
| L4.19 | Delta, gamma, theta from FD grid |


In [1]:
import numpy as np
from scipy.stats import norm
from scipy.linalg import solve_banded
import matplotlib.pyplot as plt
%matplotlib inline
np.set_printoptions(precision=6, suppress=True)


---

## 1. The problem with explicit FD (recap from Topic C)

Explicit FD requires:
$$\Delta t \le \frac{(\Delta x)^2}{\sigma^2}.$$

**Plain English:** If you want fine spatial resolution (small $\Delta x$), you are forced to take **very small time steps** too. That makes the algorithm slow.

**Implicit and Crank-Nicolson methods** remove this restriction. You pay by solving a **linear system** at each time step instead of a one-line formula.


## 2. Fully implicit FD (L4.4–L4.7)

### Model PDE (no drift, log-price)

For simplicity first ignore $\nu$ and $r$:
$$C_t + \tfrac12\sigma^2 C_{xx} = 0.$$

### Where we evaluate derivatives

| Scheme | Time derivative | Spatial derivatives |
|---|---|---|
| **Explicit** (Topic C) | $(C_n^j - C_{n+1}^j)/\Delta t$ | at time $t_{n+1}$ (newer column) |
| **Implicit** | $(C_n^j - C_{n+1}^j)/\Delta t$ | at time $t_n$ (older column — **unknown**) |

Implicit scheme:
$$
\frac{C_n^j - C_{n+1}^j}{\Delta t} + \tfrac12\sigma^2\,\frac{C_{n+1}^j - 2C_n^j + C_{n-1}^j}{(\Delta x)^2} = 0.
$$

Define $\alpha = \tfrac12\sigma^2\,\dfrac{\Delta t}{(\Delta x)^2}$. Rearrange for **unknown** $C_n^j$:
$$
\boxed{-\alpha C_n^{j+1} + (1+2\alpha)C_n^j - \alpha C_n^{j-1} = C_{n+1}^j.}
$$

### What this means operationally

- **Known:** entire column $C_{n+1}$ (from expiry, stepping back).
- **Unknown:** entire column $C_n$ — all values coupled together.
- **Solve:** a **tridiagonal** linear system $M\,\mathbf c_n = \mathbf b$.

### Unconditional stability (L4.6)

Fourier analysis gives amplification factor $|\lambda_k| \le 1$ **with no restriction** on $\Delta t$ vs $\Delta x$.

**Intuition (L4.7):** Explicit FD is like a trinomial tree — each node connects to three neighbors. Implicit FD is like each node at time $t_n$ connects to **all** nodes at $t_{n+1}$ through the coupled system. No "negative probability" problem.


## 3. Crank-Nicolson (L4.12–L4.15)

### The idea

Average the spatial operator between time $t_n$ and $t_{n+1}$:
$$
\frac{C_n^j - C_{n+1}^j}{\Delta t}
+ \tfrac12\Bigl[\mathcal L_n C_n + \mathcal L_{n+1} C_{n+1}\Bigr] = 0,
$$
where $\mathcal L$ is the spatial part of the PDE (second and first derivatives plus $-rC$).

### Why Crank-Nicolson is the favorite

| Scheme | Truncation error | Unconditionally stable? | Linear system? |
|---|---|---|---|
| Explicit | $O(\Delta t) + O(\Delta x^2)$ | No | No |
| Implicit | $O(\Delta t) + O(\Delta x^2)$ | Yes | Yes |
| **Crank-Nicolson** | $\mathbf{O(\Delta t^2) + O(\Delta x^2)}$ | Yes | Yes |

**Exam soundbite:** CN is second-order in **both** time and space, and unconditionally stable.

### General coefficients (L4.13)

For PDE $C_t + f C_{xx} + g C_x + h C = 0$, the CN update is:
$$-F_n^j C_n^{j+1} + (1+G_n^j)C_n^j - H_n^j C_n^{j-1} = F_{n+1}^j C_{n+1}^{j+1} + (1-G_{n+1}^j)C_{n+1}^j + H_{n+1}^j C_{n+1}^{j-1} + \text{(boundary terms)}$$
with
$$F_n^j = \tfrac{\Delta t}{2(\Delta x)^2}f_n^j + \tfrac{\Delta t}{4\Delta x}g_n^j,\quad
G_n^j = \tfrac{\Delta t}{(\Delta x)^2}f_n^j - \tfrac{\Delta t}{2}h_n^j,\quad
H_n^j = \tfrac{\Delta t}{2(\Delta x)^2}f_n^j - \tfrac{\Delta t}{4\Delta x}g_n^j.$$

For **log-price Black-Scholes**: $f=\tfrac12\sigma^2$, $g=\nu=r-\tfrac12\sigma^2$, $h=-r$.

You do not need to memorize $F,G,H$ for the exam if the problem gives the simplified CN equation (as Practice-Midterm P4 does).


## 4. Solving the tridiagonal system (L4.17)

Each time step: $\mathbf c_n = M^{-1}\mathbf b$ where $M$ is **tridiagonal** (only main diagonal and one off-diagonal on each side).

**Do not** invert $M$ explicitly ($O(m^3)$ waste).

**Do:** Gaussian elimination specialized to tridiagonal — **Thomas algorithm** — $O(m)$ operations.

In Python: `numpy.linalg.solve(M, b)` works for moderate grid sizes.

**Procedure each time step:**
1. Build $M$ and $\mathbf b$ from CN (or implicit) formula using known column $C_{n+1}$.
2. Apply boundary conditions (modify first/last rows or add to $\mathbf b$).
3. Solve for $\mathbf c_n$.
4. Repeat for $n = N-1, \ldots, 0$.


In [2]:
def cn_call_price(S0, K, T, r, sigma, N=400, J=400):
    # Crank-Nicolson on uniform log-price grid (same setup as Topic C explicit FD)
    nu = r - 0.5 * sigma**2
    dt = T / N
    dx = sigma * np.sqrt(3 * dt)
    x0 = np.log(S0)
    xs = x0 + np.arange(-J, J + 1) * dx
    S = np.exp(xs)
    C = np.maximum(S - K, 0.0)
    m = 2 * J - 1

    for n in range(N - 1, -1, -1):
        f = 0.5 * sigma**2
        F = 0.5 * dt / dx**2 * f + 0.25 * dt / dx * nu
        G = dt / dx**2 * f + 0.5 * dt * r   # L4.13 with h = -r
        H = 0.5 * dt / dx**2 * f - 0.25 * dt / dx * nu

        rhs = np.zeros(m)
        for k in range(m):
            j = k + 1
            rhs[k] = F * C[j + 1] + (1 - G) * C[j] + H * C[j - 1]

        tau = T - n * dt
        C[0] = 0.0
        C[-1] = S[-1] - K * np.exp(-r * tau)
        rhs[0] += H * C[0]
        rhs[-1] += F * C[-1]

        ab = np.zeros((3, m))
        ab[0, 1:] = -F
        ab[1, :] = 1 + G
        ab[2, :-1] = -H
        C[1:-1] = solve_banded((1, 1), ab, rhs)

    return C[J]

S0, K, T, r, sigma = 100, 100, 0.5, 0.05, 0.30
cn_price = cn_call_price(S0, K, T, r, sigma)
bs = S0 * norm.cdf((np.log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))) \
   - K * np.exp(-r*T) * norm.cdf((np.log(S0/K) + (r - 0.5*sigma**2)*T) / (sigma*np.sqrt(T)))
print(f"Crank-Nicolson call: {cn_price:.4f}")
print(f"Black-Scholes call:  {bs:.4f}")
print(f"Error: {abs(cn_price - bs):.4f}")
print("(Same grid as Topic C explicit FD: N=400, J=400, dx = sigma*sqrt(3*dt))")


Crank-Nicolson call: 9.6273
Black-Scholes call:  9.6349
Error: 0.0076
(Same grid as Topic C explicit FD: N=400, J=400, dx = sigma*sqrt(3*dt))


## 5. Practice-Midterm P4(b) — reconstruct hidden grid value

### Setup

PDE in **original** $S$ (not log), $r=0$:
$$\frac{\partial C}{\partial t} - \tfrac12 S^2\sigma^2\,\frac{\partial^2 C}{\partial S^2} = 0.$$

Grid: $\Delta t = 0.1$, $\Delta S = 2$, $\sigma = 0.6$. Work **backwards** from $t=1.2$ to $t=1.1$.

Partial grid ($\bar C$ = approximate solution):

| $S$ | $t=1.1$ | $t=1.2$ |
|---|---|---|
| 152 | 34.84 | 32.95 |
| 150 | **hidden** | 31.73 |
| 148 | 32.40 | 30.53 |

### Crank-Nicolson at $S=150$

Time derivative (backward from 1.2 to 1.1):
$$\frac{\bar C(150,1.2) - \bar C(150,1.1)}{\Delta t} = \frac{31.73 - C}{0.1}.$$

Spatial second derivative (central, $\Delta S=2$):
$$\frac{\partial^2 C}{\partial S^2} \approx \frac{C(S+\Delta S) - 2C(S) + C(S-\Delta S)}{(\Delta S)^2}.$$

CN averages spatial operator at $t=1.1$ and $t=1.2$:
$$
\frac{31.73 - C}{0.1} - \tfrac12(150)^2(0.6)^2 \cdot \tfrac12\Bigl[\delta_{1.2} + \delta_{1.1}\Bigr] = 0,
$$
where
$$\delta_{1.2} = \frac{32.95 - 2(31.73) + 30.53}{4} = \frac{0.02}{4} = 0.005,$$
$$\delta_{1.1} = \frac{34.84 - 2C + 32.40}{4} = \frac{67.24 - 2C}{4}.$$

Combine into one stencil (official solution form):
$$\frac{31.73 - C}{0.1} - 2025 \cdot \frac{32.95 - 2(31.73) + 30.53 - 34.84 + 2C - 32.40}{4} = 0,$$
where $2025 = \tfrac12(150)^2(0.6)^2 \cdot \tfrac12$.

Rearrange (collect $C$ terms on the left):
$$\boxed{\hat C(150,1.1) = \frac{2}{2045}\left[\frac{31.73}{0.1} - 2025\,\frac{32.95 - 2(31.73) + 30.53 - 34.84 - 32.40}{4}\right].}$$
The denominator $2045 = 1 + 2025\cdot 2/4$ comes from moving the $2C$ term to the left-hand side.


In [3]:
# Practice-Midterm P4(b) from official solution formula
# C = (2/2045) * (31.73/0.1 - 2025*(32.95 - 2*31.73 + 30.53 - 34.84 - 32.4)/4)
# Note: uses combined CN spatial stencil as in L4.12 solution

C_hidden = (2/2045) * (31.73/0.1 - 2025*(32.95 - 2*31.73 + 30.53 - 34.84 - 32.40)/4)
print(f"Reconstructed C(150, 1.1) = {C_hidden:.4f}")

# Verify by plugging into CN equation (same form as solution)
C = C_hidden
dt, dS, S, sigma = 0.1, 2, 150, 0.6
term_time = (31.73 - C) / dt
coef = 0.5 * S**2 * sigma**2 * 0.5
spatial = (32.95 - 2*31.73 + 30.53 - 34.84 + 2*C - 32.40) / (dS**2)
residual = term_time - coef * spatial
print(f"CN equation residual (should ~ 0): {residual:.6f}")


Reconstructed C(150, 1.1) = 33.5916
CN equation residual (should ~ 0): -0.000000


## 6. Practice-Midterm P4(c) — charm from the grid

### Definition

**Charm** (delta decay):
$$\text{Charm} = \frac{\partial^2 C}{\partial t\,\partial S} = \frac{\partial (\Delta)}{\partial t}.$$

**Plain English:** How fast does delta change as time passes (with $S$ held fixed)?

### FD approximation (L4.19 + solution)

1. **Delta at $t=1.2$, $S=150$** (central in $S$, both neighbors at $t=1.2$):
$$\Delta_{1.2} \approx \frac{\bar C(152,1.2) - \bar C(148,1.2)}{2\Delta S} = \frac{32.95 - 30.53}{4} = 0.605.$$

2. **Delta at $t=1.1$, $S=150$** (both neighbors at $t=1.1$):
$$\Delta_{1.1} \approx \frac{\bar C(152,1.1) - \bar C(148,1.1)}{2\Delta S} = \frac{34.84 - 32.40}{4} = 0.61.$$

3. **Time derivative of delta:**
$$\text{Charm} \approx \frac{\Delta_{1.2} - \Delta_{1.1}}{\Delta t} = \frac{0.605 - 0.61}{0.1} = \boxed{-0.05}.$$

**Exam procedure:** Write delta at each time from central differences (same time level for both neighbors), then divide by $\Delta t$.


In [4]:
delta_12 = (32.95 - 30.53) / 4
delta_11 = (34.84 - 32.40) / 4
charm = (delta_12 - delta_11) / 0.1
print(f"Delta at t=1.2: {delta_12:.4f}")
print(f"Delta at t=1.1: {delta_11:.4f}")
print(f"Charm: {charm:.4f}")


Delta at t=1.2: 0.6050
Delta at t=1.1: 0.6100
Charm: -0.0500


## 7. Greeks from the FD grid (L4.19)

At grid point $(S_j, t_n)$ with approximate value $\bar C_n^j$:

| Greek | FD formula (uniform $\Delta S$) |
|---|---|
| **Delta** | $\Delta \approx \dfrac{\bar C_n^{j+1} - \bar C_n^{j-1}}{2\Delta S}$ |
| **Gamma** | $\Gamma \approx \dfrac{\bar C_n^{j+1} - 2\bar C_n^j + \bar C_n^{j-1}}{(\Delta S)^2}$ |
| **Theta** | $\Theta \approx \dfrac{\bar C_{n+1}^j - \bar C_n^j}{\Delta t}$ |

If the grid is in **log-price** $x=\log S$:
$$\frac{\partial C}{\partial S} = \frac{1}{S}\,\frac{\partial C}{\partial x} \approx \frac{1}{S}\,\frac{\bar C_n^{j+1} - \bar C_n^{j-1}}{2\Delta x}.$$

**Exam tip:** Prefer **central** differences for delta/gamma when both neighbors exist (Practice-Midterm says so).


## 8. American options on FD grids (L4.21)

After computing **continuation value** $H_n$ from the FD scheme (implicit or CN):
$$C_n^j = \max\bigl(H_n^j,\; \text{exercise payoff at } (S_j, t_n)\bigr).$$

For a put: $C_n^j = \max(H_n^j, K - S_j)$.

This is the same `max(exercise, continue)` logic as trees (Topic B) and Longstaff-Schwartz (Topic H preview).

**HW4 P2(b):** American put under CEV dynamics — use provided `FD_CrankNicolson_Engine.price_put_CEV` with Crank-Nicolson + this max step.


## 9. Strike derivatives — Breeden–Litzenberger (L4.24–L4.25)

These link **option prices across strikes** to **risk-neutral probabilities** (also Topic 0 §14).

### Binary call from strike derivative

$$-\frac{\partial C}{\partial K}(K,T) = e^{-rT}\,\mathbb Q(S_T > K).$$

**Intuition:** A tight call spread approximates a digital payoff; its price per unit strike is $-\partial C/\partial K$.

### Density from second strike derivative

$$\frac{\partial^2 C}{\partial K^2}(K,T) = e^{-rT}\,f_{S_T}(K).$$

**FD approximation (butterfly):**
$$\frac{\partial^2 C}{\partial K^2} \approx \frac{C(K-\Delta K) - 2C(K) + C(K+\Delta K)}{(\Delta K)^2}.$$

Practice-Midterm P1(c,d) used these with market call prices.


## 10. Explicit vs Implicit vs CN — decision guide

```mermaid
flowchart TD
    start[Need to price PDE on grid] --> q1{Need unconditional stability?}
    q1 -->|No, small grid| explicit[Explicit FD - Topic C]
    q1 -->|Yes| q2{Need O dt squared accuracy?}
    q2 -->|No| implicit[Implicit FD]
    q2 -->|Yes| cn[Crank-Nicolson]
    cn --> solve[Solve tridiagonal system each step]
    implicit --> solve
```

**When in doubt on this exam:** Crank-Nicolson is the default "serious" scheme.


## 11. Exam-style exercises

**E1.** For implicit FD, why is $| \lambda_k | \le 1$ without a CFL condition?

**E2.** Write the tridiagonal equation for interior node $j$ in implicit scheme with $\alpha = \sigma^2\Delta t/(2(\Delta x)^2)$.

**E3.** From the P4 grid, compute gamma of the contract at $S=150$, $t=1.2$.

**E4.** Why does CN have $O(\Delta t^2)$ error but implicit only $O(\Delta t)$?


### Solutions

<details><summary><b>E1</b></summary>

Implicit scheme Fourier amplification $\lambda_k = 1/(1 + 4\alpha\sin^2(\pi k/(2J))) \le 1$ for all $\alpha > 0$. Errors shrink or stay bounded regardless of $\Delta t$ vs $\Delta x$.
</details>

<details><summary><b>E2</b></summary>

$-\alpha C_n^{j+1} + (1+2\alpha)C_n^j - \alpha C_n^{j-1} = C_{n+1}^j$.
</details>

<details><summary><b>E3</b></summary>

$\Gamma \approx (32.95 - 2(31.73) + 30.53)/4 = 0.02/4 = \mathbf{0.005}$.
</details>

<details><summary><b>E4</b></summary>

CN averages the spatial operator at $t_n$ and $t_{n+1}$, canceling leading $O(\Delta t)$ time error (Taylor analysis L4.15). Implicit evaluates spatial operator only at one time level.
</details>


## 12. Cheat sheet — Topic D

| # | Fact | Use |
|---|---|---|
| 1 | Implicit: $-\alpha C_n^{j+1}+(1+2\alpha)C_n^j-\alpha C_n^{j-1}=C_{n+1}^j$ | Tridiagonal solve |
| 2 | CN: average spatial op at $n$ and $n+1$ | Default scheme |
| 3 | CN error: $O(\Delta t^2)+O(\Delta x^2)$; unconditionally stable | vs explicit |
| 4 | Each step: build $M\mathbf c_n=\mathbf b$, solve tridiagonal | Algorithm |
| 5 | $\Delta \approx (C_{j+1}-C_{j-1})/(2\Delta S)$ | Greeks |
| 6 | Charm $\approx (\Delta_{t+\Delta t}-\Delta_t)/\Delta t$ | P4(c) |
| 7 | American: $C_n=\max(H_n,\text{payoff})$ | After FD step |
| 8 | $\partial^2 C/\partial K^2 = e^{-rT}f(K)$ | Breeden–Litzenberger |

---

*Next:* **Topic E — L5: Monte Carlo Foundations**
